# The Annotated Transformer

References
- The annotated transformer blogpost by Harvard NLP: https://nlp.seas.harvard.edu/annotated-transformer/
- Attention is All You Need
- Layer Normalization

## The Paper

### Overview

The paper specifically investigates the sequence transduction models which typically follow an encoder-decoder architecture based on complex recurrent or convolutional neural networks. The paper proposes a (later-found-to-be) revolutionary network architecture called the **Transformer**, which based solely on attention mechanisms and delivers SOTA performance on machine translation tasks and also shows to generalize well to other tasks.

### The Original Goal

Sequence modeling and transduction problems such as language modeling and machine translation, which typically rely on recurrent models and encoder-decoder architectures to achieve SOTA results.

### What Are Recurrent Models?

Recurrent models generate a sequence of hidden states $h_t$ as a function of the previous hidden state $h_{t - 1}$ and the input for position $t$, which has an inherently sequential nature that makes it hard for parallelization and suffers memory issues from this fundamental constraint of sequential computation.

### How Does Attention Help?

The attention mechanism has been providing concrete improvement in sequence modeling and transduction models as they allow modeling of dependencies without regard to their distance in the input or output sequences.

### The Architecture

The transformer model proposed also follows a basic encoder-decoder architecture. Specifically, the encoder maps an input sequence of symbol representations $(x_1, \cdots, x_n)$ to a sequence of continuous representations $\boldsymbol{z} = (z_1, \cdots, z_n)$. Then given the representations $\boldsymbol{z}$, the decoder generates an output sequence $(y_1, \cdots, y_m)$ of symbols one element at a time in a *autoregressive* manner, which consumes the previously generated symbols as an additional input while generating the next. The transformer follows this architecture by using *stacked* self-attention and pointwise, fully connected layers for both the encoder and the decoder, as shown in the diagram below.
The transformer model proposed also follows a basic encoder-decoder architecture. Specifically, the encoder maps an input sequence of symbol representations $(x_1, \cdots, x_n)$ to a sequence of continuous representations $\boldsymbol{z} = (z_1, \cdots, z_n)$. Then given the representations $\boldsymbol{z}$, the decoder generates an output sequence $(y_1, \cdots, y_m)$ of symbols one element at a time in a *autoregressive* manner, which consumes the previously generated symbols as an additional input while generating the next. The transformer follows this architecture by using *stacked* self-attention and pointwise, fully connected layers for both the encoder and the decoder, as shown in the diagram below.

![Transformer architecture](imgs/transformer-arch.png)

#### The Encoder and the Decoder Stacks

##### Encoder

The encoder network is composed of a stack of $N = 67$ identical layers. Each layers has two sublayers:
- The first layer is a Multi-Head Self-Attention Mechanism
- The second layer is a simple, positionwise fully connected feed-forward network.
The encoder network also incorporates a residual connection around each two sublayers, followed by layer normalization, which means that the output of each sublayer is in the format,
$$\mathtt{LayerNorm}(x + \mathtt{SubLayer}(x)),$$
where $\mathtt{SubLayer}(x)$ represents the sublayer function iteself. To facilitate these residual connections, we keep all sublayers in the model, as well as the embedding layers produce outputs of the same dimension $d_\texttt{model} = 512$.

##### Decoder

The decoder network is also composed of a stack of $N = 6$ identical layers. In addition to the two sublayers in each encoder layer, the decoder adds in a third sublayer, which performs Multi-Head Attention over the output of the encoder stack. We also incorporate residual connections around each of the sublayers followed by the layer normalization as in the encoder network. Furthermore, we modify the self-attention sublayer in the decoder stack with causality masking to prevent attending to future tokens during generation, ensure that the predictions for position $i$ depend only on the previously known outputs at positions before $i$.

#### What is Layer Normalization?

Normalization serves as a technique to modify the computations performed during training of the deep neural networks based on stochastic gradient descent algorithms to make the learning easier. Batch normalization standardizes each summed input using its mean and standard deviation across the training data, which helps feed forward neural networks converge faster even with simple SGD. In addition, the stochasticity from the batch statistics serves as a regularizer during training.

However, batch normalization requires running averages of the summed input statistics, which can be computed easily in feed forward networks as it is easy to store statistics separately for each hidden layer, but can be tricky for recurrent neural networks as the recurrent neurons often require different statistics at different timesteps due to the varying sequence length. Batch normalization also cannot be applied to online learning tasks or to extremely large distributed models where the minibatches have to be small.

##### Background: The Batch Normalization

![Batch Normalization](imgs/batch-norm.png)

A feed forward neural network is a nonlinear mapping from an input pattern $\boldsymbol{x}$ to an output vector $\boldsymbol{y}$. Consider the $l^{th}$ hidden layer, and let $a^l$ be the vector representations of the summed inputs to the neurons in that layer, we have the summed inputs are computed via a linear projection with the weight matrix $W^l$ and the bottom-up inputs $h^l$ (hidden state) as follows,
$$a^l_i = w_i^{l \top} h^l, h_i^{l + 1} = f(a_i^l + b_i^l),$$
where $f(\cdot)$ is an elementwise nonlinear function and $w_i^l$ is the incoming weights to the $i^{th}$ hidden units and $b_i^l$ is the scalar bias parameter. The parameters of the neural network can be learned using modern SGD based algorithms. Batch normalization normalizes the summed inputs to each hidden unit over the training cases. Specifically, for the $i^{th}$ summed input in the $l^{th}$ layer, the batch normalization method rescales the summed inputs according to their variances under the distribution of the data
$$\bar{a}_i^l = \frac{g_i^l}{\sigma_i^l} (a_i^l - \mu_i^l), \quad \text{where } \mu_i^l = \underset{\boldsymbol{x} \sim p(\boldsymbol{x})}{\mathbb{E}} [a_i^l],\: \sigma_i^l = \sqrt{\underset{\boldsymbol{x} \sim p(\boldsymbol{x})}{\mathbb{E}} \left[ (a_i^l - \mu_i^l)^2 \right]},$$
where $\bar{a}_i^l$ is normalized summed inputs to the $i^{th}$ hidden unit in the $l^{th}$ layer and $g_i$ is the gain parameter scaling the normalized activation before the nonlinear activation function. Here the expectation is under the whole training data distribution. Thus it is usually impractical to compute the expectations in the above equations exactly, since it would require forward passes through the whole training dataset with the current set of weights (one pass). Hence we estimate $\mu$ and $\sigma$ using the empirical samples from the current minibatch, which puts constraints on the size of the minibatch and hard to apply to recurrent neural networks.

##### The Layer Normalization

![Layer Normalization](imgs/layer-norm.png)

Now let's consider the layer normalization designed to overcome the issues of batch normalization. Since the changes in the output of one layer will tend to cause highly correlated changes in the summed inputs to the next layer, especially with ReLU like units whose outputs can change by a lot, we can consider smoothing out the loss landscape by fixing the mean and the variance of the summed inputs within each layer. Thus, we can compute the layer normalization statistics over all the hidden units in the same layer as follows,
$$\mu^l = \frac{1}{H} \sum_{i = 1}^H a_i^l,\quad \sigma^l = \sqrt{\frac{1}{H} \sum_{i = 1}^H (a_i^l - \mu^l)^2},$$
where $H$ denotes the number of hidden units in a layer. The key difference between layer normalization and batch normalization is that under layer normalization, all the hidden units in a layer share the same normalization terms $\mu$ and $\sigma$, and unlike batch normalization, the layer normalization does not impose any size constraint of the minibatch and thus can work in online learning settings with even batch size $1$.

For recurrent neural networks, where the summed inputs in the recurrent layer are computed from the curren tinput $\boldsymbol{x}^t$ and the previous vector of hidden states $\boldsymbol{h}^{t - 1}$, which are computed as,
$$\boldsymbol{a}^t = W_{hh} h^{t - 1} + W_{xh} \boldsymbol{x}^t.$$
The layer normalized recurrent layer recenters and rescales its activations as follows,
$$\boldsymbol{h}^t = f\left[ \frac{\boldsymbol{g}}{\sigma^t} \odot (\boldsymbol{a}^t - \mu^t) + \boldsymbol{b} \right],\quad \text{where } \mu^t = \frac{1}{H} \sum_{i = 1}^{H} a_i^t,\: \sigma^t = \sqrt{\frac{1}{H} \sum_{i = 1}^{H} (a_i^t - \mu^t)^2}.$$

Note that in a standard RNN, there is a tendency for the average magnitude of the summed inputs to the recurrent units to either grow or shrink at every timesetp, leading to exploding or vanishing gradients. While using layer normalizations, however, the normalization terms make it invariant to rescaling all of the summed inputs to a layer, which results in a much more stable hidden-to-hidden dynamics during training.

##### `PyTorch` Implementation

In PyTorch, the `torch.nn.LayerNorm` class implements the operation,
$$y = \frac{x - \mathbb{E} [x]}{\sqrt{\mathrm{Var}[x] + \epsilon}} \ast \gamma + \beta,$$
where $\gamma$ and $\beta$ are learnable affine transform parameters when activated.

#### Attention

Recall that the conventional neural networka architectures where the hidden activations are a linear combination of the input activations followed by a nonlinearity,
$$\boldsymbol{z} = \phi (Wv),$$
where $\boldsymbol{v} \in \mathbb{R}^v$ are the hidden feature vectors, and $W \in \mathbb{R}^{v' \times v}$ are a fixed set of weight parameters learned from the training data.

We can design a more flexible model in which we have a set of $m$ feature vectors or **values** $V \in \mathbb{R}^{m \times v}$, and the model can dynamically decide which one to use (independent of the input) based on how similar the input **query** vector $\boldsymbol{q} \in \mathbb{R}^q$ is to a set of $m$ **keys** $K \in \mathbb{R}^{m \times k}$. Then if $\boldsymbol{q}$ is most similar to key $\boldsymbol{k}_i$ at index $i$, then we use the value $\boldsymbol{v}_i$. Essentially, an attention function maps a query and a set of key-value pairs to an output, the output is computed as a weighted sum of the values, where the weight assigned to each value is computed by a compatibility function of the query with corresponding key (to measure the similarity).

##### Attention as a Soft Dictionary Lookup

The attention mechanism can be thought of as a soft dictionary lookup, where we compare the query vector $\boldsymbol{q}$ to each key $\boldsymbol{k}_i$ and then retrieve the corresponding value $\boldsymbol{v}_i$. To make this lookup operation differentiable, we compute a convex combination of the values instead of hard retrieve a single value $\boldsymbol{v}_i$,
$$\begin{aligned}
\mathtt{Attn} (\boldsymbol{q}, (\boldsymbol{k}_1, \boldsymbol{v}_1), \cdots, (\boldsymbol{k}_m, \boldsymbol{v}_m)) &= \mathtt{Attn} (\boldsymbol{q}, (\boldsymbol{k}_{1:m}, \boldsymbol{v}_{1:m})) \\
&= \sum_{i = 1}^m \alpha_i (\boldsymbol{q}, \boldsymbol{k}_{1:m}) \boldsymbol{v}_i \in \mathbb{R}^v
\end{aligned}$$
where $\alpha_i (\boldsymbol{q}, \boldsymbol{k}_{1:m})$ is the $i$-th **attention weight**, satisfying
$$0 \leq \alpha_i (\boldsymbol{q}, \boldsymbol{k}_{1:m}) \leq 1$$
for each $i$ and $\sum_i \alpha_i (\boldsymbol{q}, \boldsymbol{k}_{1:m}) = 1$ (convex combination). Note that a special case would be when all the attention weights are equal, *i.e.*, $\alpha_i (\boldsymbol{q}, \boldsymbol{k}_i) = \dfrac{1}{m}$ for all $i$, which essentially reduces to averaging pooling in deep learning. To make sure that the weights sum to $1$ for convex combination requirement, a common practice is to normalize them as follows,
$$\alpha_i (\boldsymbol{q}, \boldsymbol{k}_i) = \frac{\alpha_i (\boldsymbol{q}, \boldsymbol{k}_i)}{\sum_j \alpha_j (\boldsymbol{q}, \boldsymbol{k}_j)}$$
and to ensure that the weights are nonnegative, we adopt exponentiation, which allows us to pick *any* **attention score** function $a(\boldsymbol{q}, \boldsymbol{k}_i) \in \mathbb{R}$ to compute the similarity score of query $\boldsymbol{q}$ to key $\boldsymbol{k}_i$. Then given the attention scores, we can compute the attention weights using the SoftMax function,
$$\begin{aligned}
\alpha_i (\boldsymbol{q}, \boldsymbol{k}_{1:m}) &= \mathtt{SoftMax}_i ([a(\boldsymbol{q}, \boldsymbol{k}_1), \cdots, a(\boldsymbol{q}, \boldsymbol{k}_m)])\\
&= \frac{\exp (a (\boldsymbol{q}, \boldsymbol{k}_i))}{\sum_{j = 1}^m \exp (a (\boldsymbol{q}, \boldsymbol{k}_j))}
\end{aligned}$$

##### Parametric Attention

To make the attention mechanism scale well to large training sets or high-dimensional inputs, we adopt a parametric setting for the attention mechanism, with a fixed set of keys and values, in which we compare queries and keys in a learned embedding space. Specifically, consider the most direct approach, where we have the query $\boldsymbol{q} \in \mathbb{R}^q$ and the key $\boldsymbol{k} \in \mathbb{R}^k$ as vectors of different sizes, to map them to a common embedding space of size $h$, we compute the learnable projection $W_q \boldsymbol{q}$ and $W_k \boldsymbol{k}$, where $W_q \in \mathbb{R}^{h \times q}$ and $W_k \in \mathbb{R}^{h \times k}$, then we can pass these projected embeddings into an MLP to obtain the following **additive attention** scoring function, also called the Multilayer Perceptron Attention,
$$a(\boldsymbol{q}, \boldsymbol{k}) = \boldsymbol{w}_v^\top \mathtt{tanh} (W_q \boldsymbol{q} + W_k \boldsymbol{k}) \in \mathbb{R}.$$
A more computationally efficient approach is to assume that the queries and keys are of the same length $d$, so that we can compute $\boldsymbol{q}^\top \boldsymbol{k}$ directly. Additionally, if we assume both of the queries and keys are independent random variables with $0$ mean and unit variance, we have the mean of their inner product is $0$ and the variance would be $d$ (for independent random variables $X_i$, $\mathtt{Var}[\sum_i X_i] = \sum_i \mathtt{Var}[X_i]$). Hence, to ensure the variance of the inner product remains $1$ regardless of the size of the inputs, a conventional approach would be to divide by $\sqrt{d}$, which leads to the **Scaled Dot-Product Attention**,
$$a(\boldsymbol{q}, \boldsymbol{k}) = \boldsymbol{q}^\top \boldsymbol{k} / \sqrt{d} \in \mathbb{R}.$$
In practice, we compute the attention function on a set of queries simultaneously, which is a minibatch of $n$ vectors at a time, packing together into a query matrix $Q \in \mathbb{R}^{n \times d}$, we also pack together the keys and values into matrices $K \in \mathbb{R}^{m \times d},\: V \in \mathbb{R}^{m \times v}$, then we can compute the attention-weighted outputs,
$$\mathtt{Attn}(Q, K, V) = \mathtt{SoftMax} \left( \frac{QK^\top}{\sqrt{d}} \right) V \in \mathbb{R}^{n \times v}$$

Another reason for scaling by $1 / \sqrt{d}$ is that when the data dimension $d$ grows larger, the dot product will grow large also in magnitude, pushing the softmax function into regions where it has extremely small gradients.

##### Attention as Kernels

The attention matrix can also act like a kernel matrix. For example, consider the kernel regression problem, which is a nonparametric model of the form
$$f(x) = \sum_{i=1}^n \alpha-i (x_i, x_{1:n}) y_i$$
where $\alpha_i (x, x_{1:n}) \geq 0$ meausres the *normalized* similarity of the test input $x$ to training input $x_i$. The similarity measure used here is usually computed by defining the attention score function in terms of a *density kernel*, such as the Gaussian,
$$\mathcal{K}_\sigma (u) = \frac{1}{\sqrt{2 \pi \sigma^2}} \exp (- \frac{1}{2 \sigma^2} u^2)$$
where $\sigma$ is called the *bandwidth*. Then we can define the attention score function as
$$a(x, x_i) = \mathcal{K}_\sigma (x - x_i).$$
Since the scores are normalized, we can drop the $1 / \sqrt{2 \pi \sigma^2}$ term, and rewrite the term inside the exponential as follows,
$$\mathcal{K} (u;w) = \exp (-\frac{w^2}{2} u^2).$$
Plugging into the original model, we have
$$\begin{aligned}
f(x) &= \sum_{i = 1}^n \alpha_i (x, x_{1:n})y_i\\
    &= \sum_{i = 1}^n \frac{a(x, x_i)}{\sum_{j = 1}^n a(x, x_j)}y_i\\
    &= \sum_{i = 1}^n \frac{\mathcal{K}(x - x_i; w)}{\sum_{j = 1}^n \mathcal{K}(x - x_j;w)} y_i\\
    &= \sum_{i = 1}^n \frac{\exp \left[-\dfrac{\left(x - x_i\right)^2}{2} w^2 \right]}{\sum_{j = 1}^n \exp \left[-\dfrac{\left(x - x_j\right)^2}{2} w^2 \right]} y_i\\
    &= \sum_{i = 1}^n \mathtt{SoftMax}_i \left[-\frac{1}{2} ((x - x_1)w)^2, \cdots, -\frac{1}{2} ((x - x_n)w)^2 \right] y_i
\end{aligned}$$
which can be interpreted as a form of *nonparametric attention*, where the queries are the test points $x$, the keys are the training inputs $x_i$, and the values are the training labels $y_i$.

##### Multi-Head Attention

The idea of attention matrix as kernels naturally leads to the motivation of using multiple attention matrices at once. In fact, the paper finds out that instead of performing a single attention function with $d_\mathrm{model}$-dimensional keys, values and queries, it would be beneficial to linearly project the queries, keys and values $h$ times with different, learned linear projections to $d, d$ and $v$ dimensions, respectively. Multiple attention matrices allow us to capture different notions of similarity by letting the model to jointly attend information from different representation subspaces at different positions, leading to the basic idea of **Multi-Head Attention** (MHA) mechanism. Formally, given a query $\boldsymbol{q} \in \mathbb{R}^q$, a set of keys $\boldsymbol{k}_j \in \mathbb{R}^k$ and values $\boldsymbol{v}_j \in \mathbb{R}^v$, we can define the $i$-th attention head as,
$$\boldsymbol{h}_i = \mathtt{Attn} (W_i^{(q)} \boldsymbol{q}, \{ W_i^{(k)} \boldsymbol{k}_j, W_i^{(v)} \boldsymbol{v}_j \}) \in \mathbb{R}^{p_v}$$
where $W_i^{(q)} \in \mathbb{R}^{p_q \times q}, W_i^{(k)} \in \mathbb{R}^{p_k \times k}$, and $W_i^{(v)} \in \mathbb{R}^{p_v \times v}$ are projection matrices. We can then stack $h$ heads together and project them to the output space $\mathbb{R}^{p_o} using
$$\boldsymbol{h} = \mathtt{MultiHeadAttn}(\boldsymbol{q}, \{ \boldsymbol{k}_j, \boldsymbol{v}_j \}) = W_o \begin{bmatrix}\boldsymbol{h}_1 \\ \vdots \\ \boldsymbol{h}_h \end{bmatrix} \in \mathbb{R}^{p_o}$$
with the $\boldsymbol{h}_i$ defined as above, and $W_o \in \mathbb{R}^{p_o \times hp_v}$. Then if we set $p_q h = p_k h = p_v h = p_o$, we can compute all the output heads in parallel.

## The Code

In [1]:
# required imports
import os
from os.path import exists
import torch
import torch.nn as nn
from torch.nn.functional import log_softmax, pad
import math
import copy
import time
from torch.optim.lr_scheduler import LambdaLR
import pandas as pd
import altair as alt
from torchtext.data.functional import to_map_style_dataset
from torch.utils.data import DataLoader
from torchtext.vocab import build_vocab_from_iterator
import torchtext.datasets as datasets
import spacy
import GPUtil
import warnings
from torch.utils.data.distributed import DistributedSampler
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP

warnings.filterwarnings("ignore")
RUN_EXAMPLES = True # set to False to skip example runs

### Helpers

In [2]:
def is_interactive_notebook():
    return __name__ == "__main__"

def show_example(fn, args=[]):
    if __name__ == "__main__" and RUN_EXAMPLES:
        return fn(*args)

### The Model Architecture

Most SOTA neural sequence transduction models have an encoder-decoder architecture, where the encoder maps an input sequence of symbol representations $(x_1, \cdots, x_n)$ to a sequence of continuous representations $\boldsymbol{z} = (z_1, \cdots, z_n)$,
$$(x_1, \cdots, x_n) \quad \xmapsto{\scriptsize\mathtt{Encoder}} \quad (z_1, \cdots, z_n).$$
Then given $\boldsymbol{z}$, the decoder generates an output sequence $(y_1, \cdots, y_m)$ of symbols one element at a time. At each step the model is autoregressive, consuimg the previously generated symbols as additional input while generating the next symbol.

In [3]:
class EncoderDecoder(nn.Module):
    """
    A standard Encoder-Decoder architecture for sequence modeling and transduction.
    """
    def __init__(self, encoder, decoder, source_embedding, target_embedding, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.source_embedding = source_embedding
        self.target_embedding = target_embedding
        self.generator = generator
    
    def forward(self, source, target, source_mask, target_mask):
        """
        Forward pass to process the masked source and target sequences.
        """
        return self.decoder(
            self.encode(source, source_mask), source_mask, target, target_mask)

In [4]:
class Generator(nn.Module):
    """
    Define the Linear + Softmax generation step to produce output probabilities.
    """
    def __init__(self, d_model, vocab_size):
        super(Generator, self).__init__()
        self.proj = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        return log_softmax(self.proj(x), dim=-1)

#### The Encoder and Decoder Stacks

##### The Encoder Network

The encoder is composed of a stack of $N = 6$ identical layers, followed by layer normalization.

![The Encoder Architecture](imgs/encoder.png)

In [5]:
def layer_clone(layer, N):
    """
    Helper function to produce a stack of N identical layers.
    """
    return nn.ModuleList([copy.deepcopy(layer) for _ in range(N)])

In [6]:
class Encoder(nn.Module):
    """
    The Encoder network, which consists of a stack of N = 6 layers (per the original paper),
    followed by a normalization layer (layer normalization).
    """
    def __init__(self, layer, N):
        super(Encoder, self).__init__()
        self.layers = layer_clone(layer, N)
        self.norm = nn.LayerNorm(layer.size)
    
    def forward(self, x, mask):
        """
        Forward pass through the encoder stack of the input sequence x, with the given mask.
        """
        for layer in self.layers:
            x = layer(x, mask)
        
        return self.norm(x)

Here we implement a custom version of Layer Normalization,

In [7]:
class LayerNorm(nn.Module):
    """
    From scratch implementation of layer normalization.
    """
    def __init__(self, features, eps=1e-6):
        super(LayerNorm, self).__init__()
        self.gamma = nn.Parameter(torch.ones(features))
        self.beta = nn.Parameter(torch.zeros(features))
        self.eps = eps
    
    def forward(self, x):
        """
        Forward pass to normalize the input x.
        """
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.gamma * (x - mean) / (std + self.eps) + self.beta

We also incorporate a residual connection around each of the two sublayers before the layer normalization, and thus having the output of each sublayer,
$$\mathtt{LayerNorm} (x + \mathtt{SubLayer}(x)),$$
we also make all sublayers to produce outputs of dimension $d_\texttt{model} = 512$ to facilitate the residual connections as well as the embedding layers.

In [8]:
class SubLayerResidual(nn.Module):
    """
    A residual connection followed by a layer normalization.
    Note for code simplicity the norm is first as opposed to last.
    """
    def __init__(self, size, dropout):
        super(SubLayerResidual, self).__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, sublayer):
        """
        Apply residual connection to any sublayer with the same size.
        """
        return x + self.dropout(sublayer(self.norm(x)))

Each layer of the encoder has two sublayers, the first layer is a Multi-Head Self-Attention, and the second is a simple, positionwise fully connected feed forward network.

In [9]:
class EncoderLayer(nn.Module):
    """
    The encoder layer of the Transformer model, consisting of multi-head self-attention
    and a position-wise feed-forward network, with residual connections and layer normalization.
    """
    def __init__(self, d_model, multi_head_self_attn, feed_forward, dropout):
        super(EncoderLayer, self).__init__()
        self.multi_head_self_attn = multi_head_self_attn
        self.feed_forward = feed_forward
        self.sublayer = layer_clone(SubLayerResidual(d_model, dropout), 2)
        self.d_model = d_model
    
    def forward(self, x, mask):
        """
        Forward pass through the encoder layer with self-attention and feed-forward network.
        """
        
        # Apply multi-head self-attention sublayer
        x = self.sublayer[0](x, lambda x: self.multi_head_self_attn(x, x, x, mask))
        
        # Apply position-wise feed-forward sublayer and return
        return self.sublayer[1](x, self.feed_forward)

##### The Decoder Network

![Decoder Network](imgs/decoder.png)

The decoder network also consists of a stack of $N = 6$ identical layers.

In [10]:
class Decoder(nn.Module):
    """
    The decoder also consists of a stack of N = 6 identical layers followed
    by layer normalization.
    """
    def __init__(self, layer, N):
        super(Decoder, self).__init__()
        self.layers = layer_clone(layer, N)
        self.norm = nn.LayerNorm(layer.size)
    
    def forward(self, x, memory, source_mask, target_mask):
        """
        Forward pass through the decoder stack of the input sequence x, with the given masks.
        """
        for layer in self.layers:
            x = layer(x, memory, source_mask, target_mask)
        
        return self.norm(x)

In addition to the two sublayers in each encoder layer, the decoder also inserts a third sublayer, performing Multi-Head Attention over the output of the encoder stack. Similar to the encoder network, we also utilize the residual connections to connect each of the sublayers, followed by layer normalization.

In [11]:
class DecoderLayer(nn.Module):
    """
    The decoder layer of the Transformer model, consisting of a Masked Multi-Head Self-Attention,
    a Multi-Head Attention over the encoder output (source), and a feed forward network.
    """
    def __int__(self, size, multi_head_self_attn, multi_head_attn_source, feed_forward, dropout):
        super(DecoderLayer, self).__init__()
        self.size = size
        self.multi_head_self_attn = multi_head_self_attn
        self.multi_head_attn_source = multi_head_attn_source
        self.feed_forward = feed_forward
        self.sublayer = layer_clone(SubLayerResidual(size, dropout), 3) # three sublayers instead of two
    
    def forward(self, x, memory, source_mask, target_mask):
        """
        Forward pass through the decoder layer with masked self-attention, source attention, and feed-forward network.
        """
        m = memory # shorthand for encoder output memory
        # Apply masked multi-head self-attention sublayer
        x = self.sublayer[0](x, lambda x: self.multi_head_self_attn(x, x, x, target_mask))
        
        # Apply multi-head attention over encoder output (source) sublayer
        x = self.sublayer[1](x, lambda x: self.multi_head_attn_source(x, m, m, source_mask))
        
        # Apply position-wise feed-forward sublayer and return
        return self.sublayer[2](x, self.feed_forward)

The masking in the decoder sublayer ensures that the output embeddings are offset by the position and the predictions for position $i$ can depend only on the known outputs at positions less than $i$ (so we dont cheat).

In [12]:
def subsequent_mask(size):
    """
    Mask out subsequent positions (to prevent attending to future positions).
    """
    attn_shape = (1, size, size)
    
    # create a mask with ones in the upper triangle and zeros elsewhere
    subsequent_mask = torch.triu(torch.ones(attn_shape), diagonal=1).type(torch.uint8)
    
    # return the mask where 0 indicates allowed positions and 1 indicates masked positions
    return subsequent_mask == 0

We can visualize the mask as follows,

In [ ]:
def example_mask():
    mask = subsequent_mask(20)[0]  # (20,20) boolean tensor
    rows = [
        {"Subsequent Mask": int(mask[x, y].item()), "Window": y, "Masking": x}
        for y in range(20)
        for x in range(20)
    ]
    LS_data = pd.DataFrame(rows)

    # set gradientLength to match the chart height so the colorbar is the same length
    return (
        alt.Chart(LS_data)
        .mark_rect(stroke='white', strokeWidth=1)  # draw white grid lines between tiles
        .properties(width=500, height=500, title="Subsequent Mask Visualization")
        .encode(
            x=alt.X("Window:O", title="Target Position"),
            y=alt.Y("Masking:O", title="Source Position"),
            color=alt.Color(
                "Subsequent Mask:Q",
                scale=alt.Scale(scheme="viridis", domain=[0, 1]),
                legend=alt.Legend(
                    title="Mask (1=allowed, 0=masked)",
                    orient="right",
                    gradientLength=500,  # match chart height
                    gradientThickness=10,
                ),
            ),
        )
        .interactive()
    )

show_example(example_mask)

alt.Chart(...)

#### Attention

##### Motivation

![The Motivation](imgs/attn-motivation.png)

Note that in classical recurrent neural networks, the generated tokens might be related different source tokens, and the recurrent network learns (assumed) the representations regarding which source token contributes to which generated token implicitly. The attention mechanism is to make this process *explicit*.

##### The Attention Layer
![The Attention Layer](imgs/attn-layer.png)
An attention layer is designed specifically to make this selection process explicit. Concretely, the attention layer has a *memory* of key-value pairs and the attention output will be close to the **values** whose **keys** that are similar to the **query**. Formally, the procedure of operations inside an attention layer goes as follows,
- Given a query $\boldsymbol{q} \in \mathbb{R}^q$ and the memory $(\boldsymbol{k}_1, \boldsymbol{v}_1), \cdots, (\boldsymbol{k}_m, \boldsymbol{v}_m)$ with $\boldsymbol{k}_i \in \mathbb{R}^d,\: \boldsymbol{v}_i \in \mathbb{R}^v$.
- Compute $n$ attention scores using the corresponding attention score function, $a_1, \cdots, a_m$ by $a_i = a(\boldsymbol{q}, \boldsymbol{k}_i)$
- Use `SoftMax` to obtain the attention weights (positive and sum to $1$, aka. convex combination weights),
    $$\alpha_1, \cdots, \alpha_m = \mathtt{SoftMax}(a_1, \cdots, a_m)$$
- The output is a weighted sum of values,
    $$\mathtt{Attn}(\boldsymbol{q}, (\boldsymbol{k}_1, \boldsymbol{v}_1), \cdots, (\boldsymbol{k}_m, \boldsymbol{v}_m)) = \sum_{i = 1}^m \alpha_i \boldsymbol{v}_i \in \mathbb{R}^v$$

The attention mechanism in the attention layer described above can be summarized in the diagram below,

![The Attention Mechanism](imgs/attn-mechanism.png)

For better computing efficiency, we use the *Scaled Dot-Product Attention* as discussed above, which computes the attention output from the packed matrices $Q, K, V$, with the compute graph,

![Scaled Dot-Product Attention](imgs/scaled-dotproduct.png)

Note that, in some cases, we might want to restrict attention to a subset of the dictionary, corresponding to valid entries. For example, we might want to pad sequences to a fixed length (for efficient minibatching purposes, etc.), where we should "mask out" the padded locations, which is called **Masked Attention**. This can be implemented efficiently by setting the attention score for the masked entries to a large negative number, such as $-10^6$, so that the corresponding `SoftMax` attention weights will become effectively $0$ (in analogy to causal convolution).

In [14]:
def attention(query, key, value, mask=None, dropout=None):
    """
    Compute the Scaled Dot-Product Attention Mechanism,
    where we assume that both the queries and keys are of
    the same dimension d.
    """
    d = query.size(-1)
    attention_scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d)
    if mask is not None:
        # compute masked attention if needed, by setting masked positions to a large negative value,
        # effectively zeroing them out after the softmax
        attention_scores = attention_scores.masked_fill(mask == 0, -1e9)
    attention_weights = torch.softmax(attention_scores, dim=-1)
    if dropout is not None:
        attention_weights = dropout(attention_weights)
    return torch.matmul(attention_weights, value), attention_weights

#### Multi-Head Attention

![Multi-Head Attention](imgs/mha.png)

As discussed earlier in the paper review, the Multi-Head Attention allows the model to jointly attend to information from different representation subspaces at different positions. The results from the multiple attention heads are concatenated and then projected onto the output space, resulting in the final set of values. Using the PyTorch formulation (the affine linear transform by PyTorch linear layer is defined as $\boldsymbol{y} = \boldsymbol{x} A^\top + \boldsymbol{b}$), with the packed query, key and value matrices, $Q \in \mathbb{R}^{n \times d}, K \in \mathbb{R}^{m \times d}, V \in \mathbb{R}^{m \times v}$, we have
$$\mathtt{MultiHeadAttn}(Q, K, V) = \mathtt{concat}[\mathtt{Head}_1, \cdots, \mathtt{Head}_h] W^{(o) \top}$$
where
$$\mathtt{Head}_i = \mathtt{Attn}(Q W_i^{(q) \top}, K W_i^{(k) \top}, V W_i^{(v)\top})$$
with the project parameter matrices of the linear layer $W_i^{(q)} \in \mathbb{R}^{d_\mathrm{model} \times d}, W_i^{(k)} \in \mathbb{R}^{d_\mathrm{model} \times d}, W_i^{(v)} \in \mathbb{R}^{d_\mathrm{model} \times v}$ and $W^{(o)} \in \mathbb{R}^{hv \times d_\mathrm{model}}$.
The model proposed in the paper specfically employs $h = 8$ parallel attention layers/heads. For each of the attention heads we use $d = d_\mathrm{model} / h = 64$. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality.

In [17]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention mechanism, allowing the model to jointly attend to information
    from different representation subspaces at different positions using multiple
    attention heads.
    """
    def __init__(self, h, d_model, dropout=0.1):
        super(MultiHeadAttention, self).__init__()
        
        # make sure d_model is divisible by the number of heads h
        assert d_model % h == 0
        
        # for efficiency, we assume d_k = d_v = d = d_model // h
        self.d = d_model // h
        self.h = h  # number of heads
        
        # define linear layers for query, key, value and output projection
        self.linear_layers = layer_clone(nn.Linear(d_model, d_model), 4)
        self.attn = None
        self.dropout = nn.Dropout(p=dropout)
    
    def forward(self, query, key, value, mask=None):
        """
        Forward pass through the Multi-Head Attention mechanism.
        """
        if mask is not None:
            # same mask applied to all heads
            mask = mask.unsqueeze(1)
        
        batch_size = query.size(0)  # factor out batch size
        
        # perform linear projections and split into h heads
        query, key, value = [
            linear_layer(x).view(batch_size, -1, self.h, self.d).transpose(1, 2)
            for linear_layer, x in zip(self.linear_layers[:3], (query, key, value))
        ]
        
        # apply attention on all the projected vectors in batch
        # x has shape (batch_size, h, seq_len, d)
        x, self.attn = attention(query, key, value, mask=mask, dropout=self.dropout)
        
        # "concatenate" heads and apply final linear layer
        # x has shape (batch_size, seq_len, h * d)
        x = x.transpose(1, 2).contiguous().view(batch_size, -1, self.h * self.d)

        # delete the query, key, value matrices
        del query, key, value
        
        return self.linear_layers[3](x)

##### Applications of Attention in the Model

The Transformer model uses Multi-Head Attention in three different ways,
1. In the "encoder-decoder" architecture, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder, which allows every position in the decoder to attend over all positiions in the input sequence. This *mimics* the typical encoder-decoder attention mechanisms in *sequence-to-sequence* models.
2. The encoder contains Self-Attention layers, where all of the keys, values and queries come from the same place, which is the output of the previous layer in the encoder. Each position in the encoder can attend to all positions in the previous layer of the encoder.
3. Similarly, the Self-Attention layers in the decoder allow each position in the decoder to attend to all positions in the decoder *up to and including that position*. Also we need to prevent leftward information flow in the decoder to preserve the auto-regressive property (the attentions cannot CHEAT). We can implement this mechanism inside of the scaled dot-product attention by masking out (setting to large negative number, aka. $-\infty$) all the values in the input of the `SoftMax` function which correspond to illegal connections.

##### Position-wise Feed-Forward Networks

In addition to the Attention sublayers, each of the layers in our encoder and decoder networks contains a Fully Connected Feed-Forward Network, which is applied to *each position separately and identically*, consisting of two linear transformations with `ReLU` activation,
$$\mathtt{FFN} (\boldsymbol{x}) = \max (0, \boldsymbol{x} W_1^\top + \boldsymbol{b}_1) W_2^\top + \boldsymbol{b}_2$$

In [18]:
class PositionwiseFeedForwardNetwork(nn.Module):
    """
    A position-wise feed-forward network that applies a linear transformation
    followed by a ReLU activation, and another linear transformation.
    """
    def __init__(self, d_model, d_ffn, dropout=0.1):
        super(PositionwiseFeedForwardNetwork, self).__init__()
        
        # define the two linear transformations
        self.linear1 = nn.Linear(d_model, d_ffn)
        self.linear2 = nn.Linear(d_ffn, d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        """
        Forward pass through the Position-wise Feed-Forward Network.
        """
        return self.linear2(self.dropout(self.linear1(x).relu()))

Note that while the linear transformations are the same across different positions, they use different sets of parameters from layer to layer. Another way of describing this is as two convolutions with kernel size $1$. The dimensionality of both input and output is $d_\mathrm{model} = 512$ and the internal dimension of the network layer is $d_\mathrm{ffn} = 2048$.